# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficient page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'careers page',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'Endpoints page', 'url': 'https://endpoints.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
moonshotai/Kimi-K2.5
Updated
about 11 hours ago
•
25.4k
•
1.14k
nvidia/personaplex-7b-v1
Updated
1 day ago
•
54.5k
•
1.48k
Tongyi-MAI/Z-Image
Updated
2 days ago
•
1.92k
•
677
Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice
Updated
1 day ago
•
181k
•
789
deepseek-ai/DeepSeek-OCR-2
Updated
about 16 hours ago
•
45.3k
•
544
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.02k
Qwen3-TTS Demo
🎙
1.02k
Transform text into natural-sounding speech with custom voices
Running
on
Zero
Featured
1.27k
Qwen Image Multiple Angles 3D Camera
🎥
1.27k
Adjust cam

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 5 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nmoonshotai/Kimi-K2.5\nUpdated\nabout 11 hours ago\n•\n25.4k\n•\n1.14k\nnvidia/personaplex-7b-v1\nUpdated\n1 day ago\n•\n54.5k\n•\n1.48k\nTongyi-MAI/Z-Image\nUpdated\n2 days ago\n•\n1.92k\n•\n677\nQwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\nUpdated\n1 day ago\n•\n181k\n•\n789\ndeepseek-ai/DeepSeek-OCR-2\nUpdated\nabout 16 hours ago\n•\n45.3k\n•\n544\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nF

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face Company Brochure

---

## About Hugging Face

**Hugging Face** is the vibrant AI community and collaboration platform that is building the future of machine learning. It serves as a central hub where machine learning engineers, data scientists, and AI enthusiasts come together to create, share, and innovate on open-source models, datasets, and applications.

Hugging Face empowers the next generation of ML experts and users by providing open and ethical AI tools while fostering a global community dedicated to advancing artificial intelligence.

---

## What We Offer

### The Hugging Face Hub  
- A collaborative platform to **host, discover, and share** over **2 million machine learning models**, more than **500,000 datasets**, and **1 million+ AI applications**.  
- Supports all modalities of machine learning including **text, image, video, audio, and 3D**.  
- Enables users to build their portfolio and ML profile by publishing their work and learning from others.

### Spaces  
- A zero-setup environment to **run and share AI Apps** directly on the platform.  
- Showcases trending demos like natural-sounding text-to-speech, image editing, and 3D image manipulation powered by cutting-edge models.  

### Open-Source Stack  
- Hugging Face provides an open-source framework accelerating research and deployment in ML, enabling users to move faster from prototype to production.

### Community and Collaboration  
- The platform is built for collaboration, encouraging contributions in the form of models, datasets, and applications.  
- Facilitates vibrant discussions and knowledge exchange among thousands of community members.

---

## Our Culture

- **Open and Ethical AI**: We are committed to creating an open, transparent, and ethical AI ecosystem.  
- **Community-Driven**: Collaboration and shared knowledge are at the core of what we do.  
- **Innovation-Focused**: We encourage creative exploration and rapid innovation in AI and machine learning.  
- **Diversity and Inclusion**: Welcoming everyone passionate about machine learning, from beginners to experts, from all backgrounds.

---

## Our Customers & Users

Hugging Face supports and is trusted by a global community including:

- Machine Learning Engineers  
- AI Researchers and Scientists  
- Developers and Data Scientists  
- Academic Institutions  
- Enterprises looking to build or adopt state-of-the-art ML models  
- Open-source contributors and AI enthusiasts  

Our platform is especially popular for deploying AI models in natural language processing, computer vision, speech recognition, and multimodal applications.

---

## Careers at Hugging Face

Join us to contribute to the future of AI! Hugging Face offers exciting career opportunities for talented engineers, researchers, and community managers passionate about machine learning and open source.

- Work on cutting-edge ML technologies and shape an ethical AI future.  
- Collaborate with a diverse, passionate team and a global community.  
- Be part of a fast-growing company championing open science and innovation.

Check the Hugging Face website for current openings and application details.

---

## Connect with Hugging Face

Website: [huggingface.co](https://huggingface.co)  
Explore Models & Datasets | Join the Community | Start Building AI Apps  

**Hugging Face** — The AI community building the future.

---

*Brand Colors:* Yellow #FFD21E, Orange #FF9D00, Gray #6B7280  
*Logo assets and branding available for partners and collaborators upon request.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. Serving as a vibrant collaboration platform, Hugging Face empowers machine learning engineers, scientists, and enthusiasts worldwide to create, discover, and share open-source machine learning models, datasets, and applications. It functions as the central hub where innovation and ethical AI development come together through community effort.

---

## What We Offer

- **Models:** Browse and collaborate on over 2 million machine learning models spanning text, image, video, audio, and 3D modalities.
- **Datasets:** Access more than 500,000 datasets contributed and maintained by the community, supporting diverse AI research and applications.
- **Spaces:** Discover and deploy over 1 million AI applications built and shared on Hugging Face's interactive platform.
- **Open Source Stack:** Move faster in your ML projects with Hugging Face’s open-source tools and libraries.
- **Community Hub:** Join one of the fastest-growing AI communities to share your work, build your professional ML profile, and collaborate with peers.

---

## Company Culture

At Hugging Face, collaboration and openness are core values. The company thrives as a community-driven platform fostering ethical AI development and active knowledge sharing. Hugging Face encourages innovation, inclusion, and transparency, positioning itself as a home for the next generation of AI practitioners who believe in building an open and responsible AI future.

---

## Customers & Users

Hugging Face serves a diverse range of customers and users including:

- Machine learning engineers and researchers accelerating AI innovation.
- Enterprises looking to harness state-of-the-art AI models and tools.
- AI enthusiasts and developers building new applications on an open platform.
- Educators and students leveraging open datasets and resources for teaching and research.
- Organizations committed to ethical and open AI development.

With millions of models, datasets, and AI apps actively used and contributed to weekly, Hugging Face is a globally recognized ecosystem in machine learning.

---

## Careers and Opportunities

Hugging Face is continuously growing and looking for passionate individuals who want to shape the future of AI. Whether you are a machine learning engineer, data scientist, developer, or community organizer, Hugging Face offers the chance to work on cutting-edge technology in a vibrant, inclusive environment.

Join Hugging Face to be part of a mission-driven company dedicated to democratizing AI and building a community that empowers everyone to innovate responsibly.

Explore open positions and start your journey with Hugging Face at their Careers page.

---

## Brand Identity

- Signature Colors: Yellow (#FFD21E), Orange (#FF9D00), Gray (#6B7280)
- Logo and Brand Assets: Available in multiple formats (.svg, .png, .ai) for community and partners.

---

## Connect with Hugging Face

- Website: https://huggingface.co
- Community Forums & Docs: Engage with other ML experts and contributors
- Explore Models, Datasets, and AI Spaces directly on the platform.

---

*Join Hugging Face today — the AI community building the future!*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>